In [31]:
# HW0: robust mega acquisition (Colab-first, git-safe)
import os
import sys
import subprocess
from pathlib import Path

try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

REPO_URL = "https://github.com/egil10/stk-mat2011.git"
REPO_DIR = Path("/content/stk-mat2011") if IN_COLAB else Path.cwd().resolve().parents[1]

if IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # keep notebook reproducible if you re-run later
        subprocess.run(["git", "-C", str(REPO_DIR), "pull"], check=False)

    # optional but useful when Colab VM is fresh
    req = REPO_DIR / "requirements.txt"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req)], check=False)

# make scripts importable
SCRIPTS_DIR = REPO_DIR / "code" / "scripts"
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.append(str(SCRIPTS_DIR))

from p_duka import download_and_save

print(f"IN_COLAB={IN_COLAB}")
print(f"REPO_DIR={REPO_DIR}")
print(f"SCRIPTS_DIR={SCRIPTS_DIR}")

IN_COLAB=True
REPO_DIR=/content/stk-mat2011
SCRIPTS_DIR=/content/stk-mat2011/code/scripts


In [32]:
from datetime import datetime
import shutil
import pandas as pd

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)

# Repo output path (where p_duka writes)
repo_processed = REPO_DIR / "code" / "data" / "processed"
repo_processed.mkdir(parents=True, exist_ok=True)

# Persistent cache on Drive (survives git pull/reclone)
drive_processed = Path("/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed") if IN_COLAB else repo_processed
if IN_COLAB:
    drive_processed.mkdir(parents=True, exist_ok=True)


def month_list(start_ym: str, end_ym: str):
    """Inclusive YYYYMM range."""
    y0, m0 = int(start_ym[:4]), int(start_ym[4:6])
    y1, m1 = int(end_ym[:4]), int(end_ym[4:6])
    out = []
    y, m = y0, m0
    while (y < y1) or (y == y1 and m <= m1):
        out.append(f"{y}{m:02d}")
        if m == 12:
            y, m = y + 1, 1
        else:
            m += 1
    return out


def expected_files(pair: str, yyyymm: str):
    s = pair.replace("/", "").lower()
    return [
        f"{s}_dukascopy_bid_{yyyymm}.parquet",
        f"{s}_dukascopy_ask_{yyyymm}.parquet",
    ]


def sync_drive_to_repo():
    if not IN_COLAB:
        return
    copied = 0
    for fp in drive_processed.glob("*.parquet"):
        tgt = repo_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced Drive -> repo: {copied} new file(s)")


def sync_repo_to_drive():
    if not IN_COLAB:
        return
    copied = 0
    for fp in repo_processed.glob("*.parquet"):
        tgt = drive_processed / fp.name
        if not tgt.exists():
            shutil.copy2(fp, tgt)
            copied += 1
    print(f"Synced repo -> Drive: {copied} new file(s)")


print(f"repo_processed={repo_processed}")
print(f"drive_processed={drive_processed}")

Mounted at /content/drive
repo_processed=/content/stk-mat2011/code/data/processed
drive_processed=/content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed


In [ ]:
# Fast acquisition: newest-first + chunked + periodic sync
PAIR = "EUR/USD"

# 1) Choose one chunk per run (recommended)
# Example quick chunk:
START_YYYYMM = "201901"
END_YYYYMM   = "201912"
SYNC_EVERY = 1
MAX_FAIL_BEFORE_STOP = 3

# Pull persistent files into repo first
sync_drive_to_repo()

months = month_list(START_YYYYMM, END_YYYYMM)
months = list(reversed(months))  # newest first
print(f"Planned months: {len(months)} ({months[-1]} -> {months[0]}) newest-first")

ok, skip, fail = [], [], []
ok_since_sync = 0

for i, yyyymm in enumerate(months, 1):
    names = expected_files(PAIR, yyyymm)
    have = all((drive_processed / n).exists() or (repo_processed / n).exists() for n in names)

    if have:
        skip.append(yyyymm)
        if i % 10 == 0:
            print(f"[{i}/{len(months)}] skip {yyyymm} (already exists)")
        continue

    print(f"[{i}/{len(months)}] fetch {PAIR} {yyyymm}")
    try:
        _ = download_and_save(PAIR, yyyymm, compression="snappy")
        ok.append(yyyymm)
        ok_since_sync += 1

        # periodic sync to persistent storage
        if ok_since_sync >= SYNC_EVERY:
            sync_repo_to_drive()
            ok_since_sync = 0

    except Exception as e:
        print(f"  ERROR {yyyymm}: {e}")
        fail.append(yyyymm)

        # safety stop if endpoint is flaky
        if len(fail) >= MAX_FAIL_BEFORE_STOP:
            print(f"Stopping early: reached {MAX_FAIL_BEFORE_STOP} failures.")
            break

# final sync
sync_repo_to_drive()

print("\nDone")
print(f"Downloaded: {len(ok)}")
print(f"Skipped existing: {len(skip)}")
print(f"Failed: {len(fail)}")
if ok:
    print("Newest downloaded months:", ok[:10])   # because newest-first
if fail:
    print("Failed months:", fail[:20])

Synced Drive -> repo: 0 new file(s)
Planned months: 12 (202001 -> 202012) newest-first
[1/12] fetch EUR/USD 202012


INFO:DUKASCRIPT:current timestamp :2020-12-01T09:22:16.106000
INFO:DUKASCRIPT:current timestamp :2020-12-01T14:37:32.078000
INFO:DUKASCRIPT:current timestamp :2020-12-01T17:44:39.782000
INFO:DUKASCRIPT:current timestamp :2020-12-02T04:29:43.144000
INFO:DUKASCRIPT:current timestamp :2020-12-02T10:00:08.391000
INFO:DUKASCRIPT:current timestamp :2020-12-02T14:24:34.240000
INFO:DUKASCRIPT:current timestamp :2020-12-02T19:17:19.744000
INFO:DUKASCRIPT:current timestamp :2020-12-03T07:26:10.173000
INFO:DUKASCRIPT:current timestamp :2020-12-03T12:10:50.623000
INFO:DUKASCRIPT:current timestamp :2020-12-03T15:46:45.618000
INFO:DUKASCRIPT:current timestamp :2020-12-04T00:47:14.628000
INFO:DUKASCRIPT:current timestamp :2020-12-04T09:01:33.905000
INFO:DUKASCRIPT:current timestamp :2020-12-04T13:29:26.351000
INFO:DUKASCRIPT:current timestamp :2020-12-04T16:09:38.211000
INFO:DUKASCRIPT:current timestamp :2020-12-07T01:56:09.249000
INFO:DUKASCRIPT:current timestamp :2020-12-07T09:26:50.370000
INFO:DUK

  Received 2,063,449 ticks
Synced repo -> Drive: 2 new file(s)
[2/12] fetch EUR/USD 202011


INFO:DUKASCRIPT:current timestamp :2020-11-02T06:44:28.629000
INFO:DUKASCRIPT:current timestamp :2020-11-02T10:31:47.472000
INFO:DUKASCRIPT:current timestamp :2020-11-02T14:48:01.984000
INFO:DUKASCRIPT:current timestamp :2020-11-02T20:35:43.229000
INFO:DUKASCRIPT:current timestamp :2020-11-03T07:06:26.925000
INFO:DUKASCRIPT:current timestamp :2020-11-03T11:22:21.849000
INFO:DUKASCRIPT:current timestamp :2020-11-03T16:01:54.878000
INFO:DUKASCRIPT:current timestamp :2020-11-04T00:01:55.476000
INFO:DUKASCRIPT:current timestamp :2020-11-04T01:43:12.220000
INFO:DUKASCRIPT:current timestamp :2020-11-04T03:12:02.296000
INFO:DUKASCRIPT:current timestamp :2020-11-04T05:18:42.360000
INFO:DUKASCRIPT:current timestamp :2020-11-04T07:41:21.146000
INFO:DUKASCRIPT:current timestamp :2020-11-04T10:16:27.751000
INFO:DUKASCRIPT:current timestamp :2020-11-04T13:06:13.640000
INFO:DUKASCRIPT:current timestamp :2020-11-04T15:51:23.221000
INFO:DUKASCRIPT:current timestamp :2020-11-04T19:58:08.475000
INFO:DUK

  Received 2,447,396 ticks
Synced repo -> Drive: 2 new file(s)
[3/12] fetch EUR/USD 202010


...

  Received 5,709,997 ticks
Synced repo -> Drive: 2 new file(s)
[11/12] fetch EUR/USD 202002


INFO:DUKASCRIPT:current timestamp :2020-02-03T10:02:45.543000
INFO:DUKASCRIPT:current timestamp :2020-02-03T16:43:41.364000
INFO:DUKASCRIPT:current timestamp :2020-02-04T08:54:29.220000
INFO:DUKASCRIPT:current timestamp :2020-02-04T16:21:18.147000
INFO:DUKASCRIPT:current timestamp :2020-02-05T09:10:58.657000
INFO:DUKASCRIPT:current timestamp :2020-02-05T15:26:41.998000
INFO:DUKASCRIPT:current timestamp :2020-02-06T08:52:53.523000
INFO:DUKASCRIPT:current timestamp :2020-02-06T17:19:53.031000
INFO:DUKASCRIPT:current timestamp :2020-02-07T09:30:01.509000
INFO:DUKASCRIPT:current timestamp :2020-02-07T15:48:30.998000
INFO:DUKASCRIPT:current timestamp :2020-02-10T08:30:22.420000
INFO:DUKASCRIPT:current timestamp :2020-02-10T17:34:01.431000
INFO:DUKASCRIPT:current timestamp :2020-02-11T12:03:48.458000
INFO:DUKASCRIPT:current timestamp :2020-02-12T00:22:55.784000
INFO:DUKASCRIPT:current timestamp :2020-02-12T14:40:27.483000
INFO:DUKASCRIPT:current timestamp :2020-02-12T23:25:20.874000
INFO:DUK

  Received 1,562,078 ticks
Synced repo -> Drive: 2 new file(s)
[12/12] fetch EUR/USD 202001


INFO:DUKASCRIPT:current timestamp :2020-01-02T10:37:00.557000
INFO:DUKASCRIPT:current timestamp :2020-01-02T16:04:10.677000
INFO:DUKASCRIPT:current timestamp :2020-01-03T03:21:00.929000
INFO:DUKASCRIPT:current timestamp :2020-01-03T10:44:25.722000
INFO:DUKASCRIPT:current timestamp :2020-01-03T16:07:35.864000
INFO:DUKASCRIPT:current timestamp :2020-01-06T03:04:37.461000
INFO:DUKASCRIPT:current timestamp :2020-01-06T11:22:26.888000
INFO:DUKASCRIPT:current timestamp :2020-01-06T18:36:53.598000
INFO:DUKASCRIPT:current timestamp :2020-01-07T08:28:48.544000
INFO:DUKASCRIPT:current timestamp :2020-01-07T14:00:10.102000
INFO:DUKASCRIPT:current timestamp :2020-01-07T21:52:36.221000
INFO:DUKASCRIPT:current timestamp :2020-01-08T03:58:01.375000
INFO:DUKASCRIPT:current timestamp :2020-01-08T10:16:25.916000
INFO:DUKASCRIPT:current timestamp :2020-01-08T15:36:34.943000
INFO:DUKASCRIPT:current timestamp :2020-01-09T00:29:02.555000
INFO:DUKASCRIPT:current timestamp :2020-01-09T10:04:40.045000
INFO:DUK

  Received 1,637,755 ticks
Synced repo -> Drive: 2 new file(s)
Synced repo -> Drive: 0 new file(s)

Done
Downloaded: 12
Skipped existing: 0
Failed: 0
Newest downloaded months: ['202012', '202011', '202010', '202009', '202008', '202007', '202006', '202005', '202004', '202003']


In [34]:
# Manifest / health check
store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))
print(f"Store: {store}")
print(f"Total parquet files: {len(files)}")

total_mb = sum(f.stat().st_size for f in files) / (1024 * 1024)
print(f"Total size: {total_mb:.1f} MB")

rows = []
for fp in files:
    name = fp.name
    # expected format: eurusd_dukascopy_bid_202101.parquet
    parts = name.replace(".parquet", "").split("_")
    if len(parts) >= 4:
        yyyymm = parts[-1]
        side = parts[-2]
    else:
        yyyymm = ""
        side = ""
    rows.append({"file": name, "yyyymm": yyyymm, "side": side, "mb": fp.stat().st_size / 1e6})

m = pd.DataFrame(rows)
if len(m):
    print("\nFiles by month (head):")
    display(m.sort_values(["yyyymm", "side"]).head(20))

    month_counts = m.groupby("yyyymm").size().rename("n_files").reset_index().sort_values("yyyymm")
    print("\nMonth coverage (last 24):")
    display(month_counts.tail(24))

# Note for gitignore/git pull workflow:
# - Keep parquet in Drive path above (not in git).
# - Notebook syncs between repo output and Drive cache.
# - After git pull/reclone, data remains in Drive and is re-synced.

Store: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Total parquet files: 152
Total size: 2933.2 MB

Files by month (head):


,file,yyyymm,side,mb
0,eurusd_dukascopy_ask_202001.parquet,202001,ask,15.582485
76,eurusd_dukascopy_bid_202001.parquet,202001,bid,15.582981
1,eurusd_dukascopy_ask_202002.parquet,202002,ask,14.644880
77,eurusd_dukascopy_bid_202002.parquet,202002,bid,14.651082
2,eurusd_dukascopy_ask_202003.parquet,202003,ask,53.529266
78,eurusd_dukascopy_bid_202003.parquet,202003,bid,53.569234
3,eurusd_dukascopy_ask_202004.parquet,202004,ask,25.466486
79,eurusd_dukascopy_bid_202004.parquet,202004,bid,25.485022
4,eurusd_dukascopy_ask_202005.parquet,202005,ask,19.240871
80,eurusd_dukascopy_bid_202005.parquet,202005,bid,19.255873



Month coverage (last 24):


,yyyymm,n_files
52,202405,2
53,202406,2
54,202407,2
55,202408,2
56,202409,2
57,202410,2
58,202411,2
59,202412,2
60,202501,2
61,202502,2


In [35]:
# End-of-HW0 summary: Drive processed folder + month coverage
import re
from pathlib import Path

import pandas as pd

store = drive_processed if IN_COLAB else repo_processed
files = sorted(store.glob("*.parquet"))

# Filename pattern: eurusd_dukascopy_bid_202101.parquet
pat = re.compile(
    r"^(?P<sym>.+)_dukascopy_(?P<side>bid|ask)_(?P<ym>\d{6})\.parquet$",
    re.I,
)

rows = []
for fp in files:
    m = pat.match(fp.name)
    if not m:
        rows.append(
            {
                "file": fp.name,
                "symbol": "",
                "side": "",
                "yyyymm": "",
                "rows": None,
                "mb": fp.stat().st_size / 1e6,
            }
        )
        continue
    try:
        import pyarrow.parquet as pq
        nr = pq.ParquetFile(fp).metadata.num_rows
    except Exception:
        nr = len(pd.read_parquet(fp, columns=["datetime"]))

    rows.append(
        {
            "file": fp.name,
            "symbol": m.group("sym").upper(),
            "side": m.group("side").lower(),
            "yyyymm": m.group("ym"),
            "rows": int(nr),
            "mb": fp.stat().st_size / 1e6,
        }
    )

meta = pd.DataFrame(rows)
parsed = meta[meta["yyyymm"] != ""].copy()

total_bytes = sum(f.stat().st_size for f in files)
n_files = len(files)

print("=" * 60)
print("HW0 — processed store summary")
print("=" * 60)
print(f"Folder: {store}")
print(f"Parquet files: {n_files}")
print(f"Total size: {total_bytes / (1024**3):.3f} GB  ({total_bytes / (1024**2):.1f} MB)")

if len(parsed):
    bid = parsed[parsed["side"] == "bid"].rename(columns={"rows": "rows_bid", "mb": "mb_bid"})
    ask = parsed[parsed["side"] == "ask"].rename(columns={"rows": "rows_ask", "mb": "mb_ask"})
    cov = (
        bid[["symbol", "yyyymm", "rows_bid", "mb_bid", "file"]]
        .merge(
            ask[["symbol", "yyyymm", "rows_ask", "mb_ask", "file"]],
            on=["symbol", "yyyymm"],
            how="outer",
            suffixes=("_bidfile", "_askfile"),
        )
        .sort_values(["symbol", "yyyymm"])
    )
    cov["has_bid"] = cov["rows_bid"].notna()
    cov["has_ask"] = cov["rows_ask"].notna()
    cov["complete_pair"] = cov["has_bid"] & cov["has_ask"]
    cov["year"] = cov["yyyymm"].str[:4]

    rows_bid = int(cov["rows_bid"].fillna(0).sum())
    rows_ask = int(cov["rows_ask"].fillna(0).sum())

    print()
    print(f"Rows bid side (sum of bid files): {rows_bid:,}")
    print(f"Rows ask side (sum of ask files): {rows_ask:,}")
    print(f"Rows all files summed:            {rows_bid + rows_ask:,}")
    print()
    print(
        f"Months with complete bid+ask: {int(cov['complete_pair'].sum())} "
        f"/ {len(cov)} rows in coverage table"
    )
    print(f"Months missing bid: {int((~cov['has_bid']).sum())}")
    print(f"Months missing ask: {int((~cov['has_ask']).sum())}")

    print("\n--- By year (complete bid+ask months) ---")
    byy = (
        cov.groupby("year")
        .agg(
            months_total=("yyyymm", "nunique"),
            months_complete=("complete_pair", "sum"),
            rows_bid=("rows_bid", "sum"),
            rows_ask=("rows_ask", "sum"),
        )
        .sort_index()
    )
    display(byy)

    print("\n--- Month coverage (symbol, yyyymm) — tail ---")
    display(
        cov[
            [
                "symbol",
                "yyyymm",
                "has_bid",
                "has_ask",
                "complete_pair",
                "rows_bid",
                "rows_ask",
            ]
        ].tail(36)
    )

    # Optional: list gaps inside [min_ym, max_ym] for each symbol
    print("\n--- Missing YYYYMM inside observed range (per symbol) ---")
    for sym, g in cov.groupby("symbol"):
        yms = sorted(g["yyyymm"].unique())
        if not yms:
            continue
        lo, hi = yms[0], yms[-1]

        def nxt(ym):
            y, m = int(ym[:4]), int(ym[4:6])
            if m == 12:
                return f"{y+1}01"
            return f"{y}{m+1:02d}"

        have = set(yms)
        missing = []
        cur = lo
        while cur <= hi:
            if cur not in have:
                missing.append(cur)
            if cur == hi:
                break
            cur = nxt(cur)
        print(f"{sym}: range {lo}..{hi} | missing inside range: {len(missing)}")
        if missing:
            print("  ", missing[:40], ("..." if len(missing) > 40 else ""))

else:
    print("\n(No Dukascopy-style filenames parsed — check naming.)")

print()
print("Approx. unique tick times: use bid row sum when bid/ask are paired.")
print("=" * 60)

HW0 — processed store summary
Folder: /content/drive/MyDrive/GITHUB-COLAB/stk-mat2011/data/processed
Parquet files: 152
Total size: 2.864 GB  (2933.2 MB)

Rows bid side (sum of bid files): 163,888,025
Rows ask side (sum of ask files): 163,888,025
Rows all files summed:            327,776,050

Months with complete bid+ask: 76 / 76 rows in coverage table
Months missing bid: 0
Months missing ask: 0

--- By year (complete bid+ask months) ---


,months_total,months_complete,rows_bid,rows_ask
year,,,,
2020,12,12,32763984,32763984
2021,12,12,16797709,16797709
2022,12,12,36950672,36950672
2023,12,12,27553506,27553506
2024,12,12,20690554,20690554
2025,12,12,23463391,23463391
2026,4,4,5668209,5668209



--- Month coverage (symbol, yyyymm) — tail ---


,symbol,yyyymm,has_bid,has_ask,complete_pair,rows_bid,rows_ask
40,EURUSD,202305,True,True,True,2236715,2236715
41,EURUSD,202306,True,True,True,2051718,2051718
42,EURUSD,202307,True,True,True,2114645,2114645
43,EURUSD,202308,True,True,True,2238800,2238800
44,EURUSD,202309,True,True,True,1737379,1737379
45,EURUSD,202310,True,True,True,2199274,2199274
46,EURUSD,202311,True,True,True,1938713,1938713
47,EURUSD,202312,True,True,True,2113635,2113635
48,EURUSD,202401,True,True,True,2173014,2173014
49,EURUSD,202402,True,True,True,1709509,1709509



--- Missing YYYYMM inside observed range (per symbol) ---
EURUSD: range 202001..202604 | missing inside range: 0

Approx. unique tick times: use bid row sum when bid/ask are paired.
